# C3 Step 7 — three experiments in one session (Colab)

Reuses `E005_upload.tgz` (already in `C3 Pilot\`). Run cells top to bottom.
Everything is protocol v1 (no speaker count unless the cell SAYS oracle,
overlap included, explicit UEM, collars 0.0/0.25) and checkpointed — if Colab
dies, re-run from Cell 3 and finished work is skipped.

| Exp | What | Question it answers |
|---|---|---|
| **E005b** | sweep clustering `Fa`,`Fb` | does ANY exposed knob move DER? (seals the E005 null) |
| **E006** | oracle speaker count, labelled | is the failure counting, or can the embeddings not separate the voices at all? |
| **E007** | swap the speaker-embedding model | does a different pretrained embedding transfer better to Sinhala-English? |

**Labelling rule (trap #2):** E006 numbers are an ORACLE condition. They are
diagnosis, never headline. The zero-shot baseline remains "the number".

Setup as before: T4 GPU, Colab secret `HF_TOKEN`, community-1 terms accepted.
E007 also needs the terms for `pyannote/wespeaker-voxceleb-resnet34-LM` and
`pyannote/segmentation-3.0` accepted on huggingface.co (one click each).
Budget: roughly 1.5–2 h of T4 total.


In [ ]:
# Cell 1 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Cell 2 — install (2-3 min), then Runtime > Restart session, skip to Cell 3
!pip install -q pyannote.audio pyannote.metrics
print('NOW: Runtime > Restart session, then continue from Cell 3.')


In [ ]:
# Cell 3 — upload E005_upload.tgz (same file as last time), then run
import os, glob
if not os.path.exists('/content/E005_upload.tgz'):
    raise SystemExit('Upload E005_upload.tgz first (folder icon -> upload).')
!cd /content && tar xf E005_upload.tgz

TEST = ['IRD_Harshana_Silva', 'IRD_Recording_2_IIT']   # real audio. FROZEN.
truths = sorted(os.path.basename(p)[:-5] for p in glob.glob('/content/rttm_truth/*.rttm'))
TUNE = [c for c in truths if c not in TEST]
assert len(truths) == 27, truths
print(f'{len(TUNE)} tune clips, {len(TEST)} test clips')


In [ ]:
# Cell 4 — model + scoring helpers (identical math to E004/E005)
import os
os.environ['PYANNOTE_METRICS_ENABLED'] = '0'
import torch
from google.colab import userdata
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-community-1',
                                    token=userdata.get('HF_TOKEN'))
pipeline.to(torch.device('cuda'))
DEFAULTS = pipeline.parameters(instantiated=True)
print('defaults:', DEFAULTS)

def load_rttm(path, uri):
    ann = Annotation(uri=uri)
    for i, line in enumerate(open(path, encoding='utf-8')):
        p = line.split()
        if len(p) >= 8 and p[0] == 'SPEAKER' and float(p[4]) > 0:
            ann[Segment(float(p[3]), float(p[3]) + float(p[4])), f't{i}'] = p[7]
    return ann
def load_uem(path, uri):
    tl = Timeline(uri=uri)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) >= 4: tl.add(Segment(float(p[2]), float(p[3])))
    return tl

TRUTH = {c: (load_rttm(f'/content/rttm_truth/{c}.rttm', c),
             load_uem(f'/content/rttm_truth/{c}.uem', c)) for c in truths}
NSPK  = {c: len(TRUTH[c][0].labels()) for c in truths}   # true speaker counts

def score(hyps, clips, collar=0.0):
    pooled = dict.fromkeys(('confusion', 'missed detection', 'false alarm', 'total'), 0.0)
    per = {}
    for c in clips:
        ref, uem = TRUTH[c]
        comp = DiarizationErrorRate(collar=collar, skip_overlap=False)(
            ref, hyps[c], uem=uem, detailed=True)
        per[c] = sum(comp[k] for k in ('confusion','missed detection','false alarm'))/comp['total']*100
        for k in pooled: pooled[k] += comp[k]
    der  = sum(pooled[k] for k in ('confusion','missed detection','false alarm'))/pooled['total']*100
    conf = pooled['confusion']/pooled['total']*100
    return der, conf, per

# fixed reference points (E004/E005, VERIFIED):
print('Zero-shot references — TUNE pooled ~20.05-20.15 @0.0; TEST pooled 7.50 '
      '(Harshana 27.99, IIT 4.87)')


In [ ]:
# Cell 5 — E005b: Fa/Fb sweep on TUNE (checkpointed)
import copy, csv, time, os
CL = next(k for k, v in DEFAULTS.items() if isinstance(v, dict) and 'Fa' in v)
Fa0, Fb0 = DEFAULTS[CL]['Fa'], DEFAULTS[CL]['Fb']
GRID = [(fa, fb) for fa in (Fa0/4, Fa0/2, Fa0, Fa0*2, Fa0*4)
                 for fb in (Fb0/2, Fb0, Fb0*2)]
CKPT = '/content/e005b_grid.csv'
done = set()
if os.path.exists(CKPT):
    done = {(float(r['Fa']), float(r['Fb'])) for r in csv.DictReader(open(CKPT))}
else:
    with open(CKPT, 'w', newline='') as fh:
        csv.writer(fh).writerow(['Fa', 'Fb', 'DER_tune_c0.0', 'conf'])

for n, (fa, fb) in enumerate(GRID, 1):
    if (round(fa, 6), round(fb, 6)) in done or (fa, fb) in done: continue
    params = copy.deepcopy(DEFAULTS); params[CL]['Fa'] = fa; params[CL]['Fb'] = fb
    pipeline.instantiate(params)
    t0 = time.time()
    hyps = {c: pipeline(f'/content/processed/{c}.wav').speaker_diarization for c in TUNE}
    der, conf, _ = score(hyps, TUNE)
    with open(CKPT, 'a', newline='') as fh:
        csv.writer(fh).writerow([fa, fb, f'{der:.4f}', f'{conf:.4f}'])
    print(f'[{n}/{len(GRID)}] Fa={fa:.4g} Fb={fb:.4g}  DER {der:.2f}  conf {conf:.2f}  ({time.time()-t0:.0f}s)')

rows = sorted(csv.DictReader(open(CKPT)), key=lambda r: float(r['DER_tune_c0.0']))
print('\nbest:', rows[0], '\ndefault Fa/Fb:', Fa0, Fb0)
print('If best beats default by <0.5 pp: the E005 null is sealed — no knob moves it.')
BEST_FAFB = rows[0]


In [ ]:
# Cell 6 — E005b on TEST only if the winner actually beat the default
import copy
if float(BEST_FAFB['DER_tune_c0.0']) < 19.5:      # >0.5 pp better than ~20.05
    params = copy.deepcopy(DEFAULTS)
    params[CL]['Fa'] = float(BEST_FAFB['Fa']); params[CL]['Fb'] = float(BEST_FAFB['Fb'])
    pipeline.instantiate(params)
    hyps = {c: pipeline(f'/content/processed/{c}.wav').speaker_diarization for c in TEST}
    for collar in (0.0, 0.25):
        der, conf, per = score(hyps, TEST, collar)
        print(f'E005b TEST collar {collar}: pooled {der:.2f} (before 7.50/6.66)  {per}')
    import os
    os.makedirs('/content/rttm_e005b', exist_ok=True)
    for c in TEST:
        with open(f'/content/rttm_e005b/{c}.rttm', 'w') as fh: hyps[c].write_rttm(fh)
else:
    print('Winner within 0.5 pp of default -> null sealed, TEST run skipped '
          '(protocol: do not touch TEST for a non-improvement).')


In [ ]:
# Cell 7 — E006: ORACLE speaker count (labelled diagnostic, never a headline)
import csv, time
pipeline.instantiate(DEFAULTS)                     # back to shipped defaults
CKPT = '/content/e006_oracle.csv'
import os
done = set()
if os.path.exists(CKPT):
    done = {r['clip'] for r in csv.DictReader(open(CKPT))}
else:
    with open(CKPT, 'w', newline='') as fh:
        csv.writer(fh).writerow(['clip', 'condition', 'DER_c0.0'])

oracle_hyps = {}
for n, c in enumerate(truths, 1):
    if c in done: continue
    t0 = time.time()
    ann = pipeline(f'/content/processed/{c}.wav',
                   num_speakers=NSPK[c]).speaker_diarization   # <-- THE ORACLE
    oracle_hyps[c] = ann
    der, _, _ = score({c: ann}, [c])
    with open(CKPT, 'a', newline='') as fh:
        csv.writer(fh).writerow([c, f'oracle_n={NSPK[c]}', f'{der:.4f}'])
    print(f'[{n}/{len(truths)}] {c}: oracle DER {der:.2f}  ({time.time()-t0:.0f}s)')

if oracle_hyps:
    os.makedirs('/content/rttm_e006_oracle', exist_ok=True)
    for c, ann in oracle_hyps.items():
        with open(f'/content/rttm_e006_oracle/{c}.rttm', 'w') as fh: ann.write_rttm(fh)
    for name, clips in (('TUNE', TUNE), ('TEST', TEST)):
        if all(c in oracle_hyps for c in clips):
            der, conf, per = score({c: oracle_hyps[c] for c in clips}, clips)
            print(f'\nE006 ORACLE pooled {name}: DER {der:.2f} (conf {conf:.2f})')
    print('\nREADING: zero-shot TUNE was ~20.1. If oracle DER collapsed (<10): '
          'embeddings are OK, the failure is COUNTING/clustering. If it stayed '
          'high (>15): the embeddings cannot separate these voices -> the '
          'strongest possible case for embedding fine-tuning (S3).')


In [ ]:
# Cell 8 — E007: swap the speaker-embedding model (zero-shot comparison)
# Builds a classic SpeakerDiarization pipeline (segmentation-3.0 + a chosen
# embedding) with the published 3.1 parameters. This is BOTH a 3.1-style
# reconstruction and an embedding A/B. If construction fails on this library
# version, paste the full error into the chat — do not guess at fixes.
import os, csv, time
from pyannote.audio.pipelines import SpeakerDiarization as SDPipeline
from google.colab import userdata

EMBEDDINGS = ['pyannote/wespeaker-voxceleb-resnet34-LM']   # add more ids to compare
CANDIDATES = [
    {'segmentation': {'min_duration_off': 0.0},
     'clustering': {'threshold': 0.6, 'Fa': 0.07, 'Fb': 0.8}},            # 4.x style
    {'segmentation': {'min_duration_off': 0.0},
     'clustering': {'method': 'centroid', 'min_cluster_size': 12,
                    'threshold': 0.7045654963945799}},                     # 3.1 style
]

CKPT = '/content/e007_embeddings.csv'
done = set()
if os.path.exists(CKPT):
    done = {(r['embedding'], r['set']) for r in csv.DictReader(open(CKPT))}
else:
    with open(CKPT, 'w', newline='') as fh:
        csv.writer(fh).writerow(['embedding', 'set', 'DER_c0.0', 'conf', 'per_clip'])

for emb in EMBEDDINGS:
    try:
        import inspect
        _p = inspect.signature(SDPipeline.__init__).parameters
        _tk = ({'token': userdata.get('HF_TOKEN')} if 'token' in _p else
               {'use_auth_token': userdata.get('HF_TOKEN')} if 'use_auth_token' in _p else {})
        alt = SDPipeline(segmentation='pyannote/segmentation-3.0',
                         embedding=emb, **_tk)
        _ok = False
        try:
            alt.instantiate(alt.default_parameters()); _ok = True
        except Exception:
            for _cand in CANDIDATES:
                try:
                    alt.instantiate(_cand); _ok = True; break
                except ValueError: pass
        if not _ok:
            print('no parameter shape fit; search space:', alt.parameters()); continue
        import torch; alt.to(torch.device('cuda'))
    except Exception as e:
        print(f'COULD NOT BUILD pipeline with {emb} on this version:\n{e!r}\n'
              'Paste this error into the chat.'); continue
    for name, clips in (('TUNE', TUNE), ('TEST', TEST)):
        if (emb, name) in done: continue
        t0 = time.time()
        hyps = {c: alt(f'/content/processed/{c}.wav') for c in clips}
        hyps = {c: (h.speaker_diarization if hasattr(h, 'speaker_diarization') else h)
                for c, h in hyps.items()}
        der, conf, per = score(hyps, clips)
        with open(CKPT, 'a', newline='') as fh:
            csv.writer(fh).writerow([emb, name, f'{der:.4f}', f'{conf:.4f}', per])
        print(f'{emb} on {name}: DER {der:.2f} (conf {conf:.2f})  ({time.time()-t0:.0f}s)')
        if name == 'TEST':
            os.makedirs('/content/rttm_e007', exist_ok=True)
            for c in clips:
                with open(f'/content/rttm_e007/{c}.rttm', 'w') as fh:
                    hyps[c].write_rttm(fh)
print('\nCompare with community-1: TUNE ~20.1 / TEST 7.50. Lower = that '
      'embedding transfers better to Sinhala-English.')


In [ ]:
# Cell 9 — download everything
!cd /content && zip -qr step7_results.zip e005b_grid.csv e006_oracle.csv e007_embeddings.csv rttm_e006_oracle rttm_e007 rttm_e005b 2>/dev/null; true
from google.colab import files
files.download('/content/step7_results.zip')
# Unzip into H:\Research Project\C3 Pilot\results\ (a step7 folder is fine),
# send the zip to the chat, and the logbook entries get written from it.
